# CineScore V6 Phase 3: Predictive Modeling (The Regularized Tournament)
We have abandoned Target Leakage. This notebook utilizes native One-Hot Encoding for our restricted dimensionality (Big 50 Studios, Top 8 Genres) and forces strict `max_depth` regularization on tree models to prevent catastrophic overfitting.

In [ ]:
%pip install -q xgboost lightgbm


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor  # type: ignore
from lightgbm import LGBMRegressor  # type: ignore
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib
import warnings
import IPython

warnings.filterwarnings('ignore')

DATA_PATH = "v6_master_analytical_df.csv"
try:
    from google.colab import files
    print("🌐 Google Colab Web detected! Please upload 'v6_master_analytical_df.csv'.")
    uploaded = files.upload()
    if uploaded:
        DATA_PATH = list(uploaded.keys())[0]
        print(f"\n✅ Successfully locked dataset to: {DATA_PATH}")
except ImportError:
    print("💻 Running outside Colab Web. Defaulting to local relative paths...")
    DATA_PATH = "../Data/Processed_Dataset/v6_master_analytical_df.csv"

df = pd.read_csv(DATA_PATH)


### Step 1: Base Target Extraction & Safe One-Hot Encoding

In [ ]:
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year

def apply_cpi_multiplier(year):
    if pd.isna(year):
        return 1.0
    if year < 1990:
        return 3.5
    if year < 2000:
        return 2.2
    if year < 2010:
        return 1.6
    if year < 2020:
        return 1.2
    return 1.0

df['cpi_multiplier'] = df['release_year'].apply(apply_cpi_multiplier)
df['inflated_budget'] = df['budget'] * df['cpi_multiplier']
df['inflated_revenue'] = df['revenue'] * df['cpi_multiplier']

df['log_inflated_revenue'] = np.log1p(df['inflated_revenue'])
df['log_inflated_budget'] = np.log1p(df['inflated_budget'])

# Dynamically reconstruct Hollywood Gravity Flag natively
if 'original_language' in df.columns:
    df['is_english'] = (df['original_language'] == 'en').astype(int)
else:
    df['is_english'] = 1

target = 'log_inflated_revenue'

numeric_features = [
    'log_inflated_budget', 'runtime', 'release_year', 
    'actor_1_hpi', 'actor_2_hpi', 'actor_3_hpi', 
    'director_hpi', 'producer_hpi', 'writer_hpi',
    'cast_synergy_mult', 'crew_synergy_mult', 'lead_duo_synergy_mult',
    'corenswet_imputation', 'is_english', 'is_epic_window', 'is_franchise',
    'four_quadrant_appeal', 'high_concept_marketability'  # <-- Don't forget these two!
]


nominal_features = ['primary_genre', 'primary_studio', 'release_season']

ml_df = df[numeric_features + nominal_features + [target]].dropna()

# V6 STRICT ONE-HOT ENCODING (Zero Leakage)
X_encoded = pd.get_dummies(ml_df[numeric_features + nominal_features], columns=nominal_features)
y = ml_df[target]

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

active_features = [col for col in X_train.columns]
print(f"Final OHE Training Set: {X_train.shape[0]} | Testing Set: {X_test.shape[0]}")
print(f"Active OHE Vector Dimensionality: {len(active_features)}")


### Step 2: CPU-Bound Tournament with Strict Tree Regularization

In [ ]:
models = {
    'RandomForest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
}

# V6 Constraint: Caps on Max Depth to strictly ban High-Cardinality Overfitting
param_grids = {
    'RandomForest': {
        'n_estimators': [50, 100, 200, 300],
        'max_depth': [5, 8, 10]  
    },
    'XGBoost': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [5, 8, 10]
    },
    'LightGBM': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [5, 8, 10]
    }
}

results = []
best_r2 = -float('inf')
champion_model = None
champion_name = ""

print("Starting Protected Array Tournament...")
for name, model in models.items():
    print(f"Tuning {name} on Regularized Array...")
    search = RandomizedSearchCV(
        model, 
        param_distributions=param_grids[name], 
        n_iter=15, 
        cv=3, 
        scoring='r2', 
        random_state=42, 
        n_jobs=-1
    )
    search.fit(X_train, y_train)
    best_estimator = search.best_estimator_
    
    log_preds = best_estimator.predict(X_test)
    true_revenue = np.expm1(y_test)
    pred_revenue = np.expm1(log_preds)
    
    r2_log = r2_score(y_test, log_preds) 
    r2_linear = r2_score(true_revenue, pred_revenue)
    mae = mean_absolute_error(true_revenue, pred_revenue)

    results.append({'Model': name, 'R2 (Log)': r2_log, 'R2 (Linear)': r2_linear, 'MAE ($)': mae})
    
    if r2_log > best_r2:
        best_r2 = r2_log
        champion_model = best_estimator
        champion_name = name

leaderboard = pd.DataFrame(results).sort_values('R2 (Log)', ascending=False)
print("\n--- TOURNAMENT RESULTS ---")
IPython.display.display(leaderboard)


### Step 3: The Meta-Ensemble Stacking Regressor
Fusing the macro-variance of Random Forest, the error-correction of LightGBM, and the dense mapping of XGBoost under a Ridge Regression Meta-Learner.

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV

print("Initiating Level 2 Meta-Ensemble Stacking Architecture...")

# We strictly enforce the parameters proven optimal during the V6 Tournament hyper-sweeps
estimators = [
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)),
    ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1)),
    ('lgbm', LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1, verbose=-1))
]

# The Meta-Learner utilizes Ridge to massively penalize the base models if they overfit against each other
stacking_regressor = StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV(), 
    cv=5,
    n_jobs=-1
)

print("Training Meta-Model (This may take roughly 20 to 60 seconds)...")
stacking_regressor.fit(X_train, y_train)

stack_preds = stacking_regressor.predict(X_test)
stack_true_revenue = np.expm1(y_test)
stack_pred_revenue = np.expm1(stack_preds)

r2_stack_log = r2_score(y_test, stack_preds)
r2_stack_lin = r2_score(stack_true_revenue, stack_pred_revenue)
mae_stack = mean_absolute_error(stack_true_revenue, stack_pred_revenue)

print(f"\n🚀 --- ENSEMBLE STACKING REGRESSOR RESULTS --- 🚀")
print(f"R2 (Log Base): {r2_stack_log:.4f}")
print(f"R2 (Linear):   {r2_stack_lin:.4f}")
print(f"MAE Error ($): ${mae_stack:,.2f}")

# Persist the Oracle brain for deployment
os.makedirs('../Data/Processed_Dataset/', exist_ok=True)
try:
    joblib.dump(stacking_regressor, '../Data/Processed_Dataset/v6_cinescore_oracle_stack.pkl')
    print("\nOracle Brain safely exported to v6_cinescore_oracle_stack.pkl")
except Exception as e:
    print(f"Oracle Export failed (likely a Google Colab local path collision). Error: {e}")
